# Beyond top degree

Round 1's feature rules choose each seed among the 2j highest-degree unused nodes, so they share 50–100% of their seeds with top degree. This notebook asks two questions on the pairs we already have:
1. **Is the best rule near top degree, or somewhere else?** Each feature is followed along a path away from top degree.
2. **Does a feature help on its own, or only through degree?** Each rule far from top degree is compared with a **look-alike**: random nodes with the same degrees.

**The path.** Seed j is chosen among the m·j highest-degree unused nodes, for m = 2 (round 1), 8 and all nodes; percolation adds 4 and 16. With m = 1 a rule is essentially top degree, so every path starts there.

| Objective | Picks seeds that maximise | Run for |
|---|---|---|
| pair mass | seed pairs sharing a non-seed neighbour (round 1's clustered target) | percolation, GOAT (not Chung–Lu), FAQ |
| uniqueness | how evenly the seeds split nodes into fingerprint groups (fingerprint = capped distances to the seeds); for FAQ, unique fingerprints first | all |
| frozen model | the logistic score fitted on the fixed-pair random sets | percolation, GOAT |
| witness score | expected number of non-seeds with at least two seed edges kept in B (theory, no fitted weights) | percolation |

**Look-alike.** At each position, a random node with the same degree as the rule's seed. The rule's gain over random then splits into two parts:
- **look-alike − random:** what that degree level buys;
- **rule − look-alike:** what the feature adds on top.

**Reused results.** Top degree, random and the round-1 rules come from round 1, `hub_test` and `borderline_check` on the same pairs, with the same seeds. At 2j, pair mass is round 1's clustered rule.

| Arm | Pairs |
|---|---|
| Percolation | 20 round-1 pairs per model, plus `borderline_check`'s 50 fresh pairs on ER and PAPER (random is run on those) |
| GOAT | 20 round-1 pairs at round 1's regulariser (ER 100, PAPER 300, Chung–Lu 300) |
| GOAT, other regulariser | the same pairs at ER 300, PAPER 100 and Chung–Lu 100; top degree, random and hub-free top degree are rerun there |
| SGM (FAQ) | 20 round-1 pairs, ER and PAPER (on Chung–Lu neither feature moves away from top degree) |

**How to read the path plots** (written before the runs)
- **Lowest at "all nodes", and the rule beats its look-alike:** the feature drives the gain, and the best rule is far from top degree.
- **Lowest in the middle:** keep some degree; the low point is the best mix.
- **Rising as the pool widens, and the rule ≈ its look-alike:** degree drives the gain, and the feature only breaks ties.

Guess before running: percolation bottoms out around 2j–8j, GOAT on Chung–Lu at 8j or all nodes, and FAQ near top degree.

**Run time** (3 pairs at a time on the Mac): percolation about 45 minutes, GOAT about 2 hours (both regularisers), FAQ about 2 hours. Each pair is saved when it finishes, so the run can be stopped and resumed, and the analysis uses whatever is done.

| Section | Content |
|---|---|
| 1 | Checks |
| 2 | Where each rule's seeds sit |
| 3 | The paths (main result) |
| 4 | Degree level or feature? |
| 5 | Success curves |
| 6 | Pair-by-pair comparisons |
| 7 | GOAT at both regularisers |
| 8 | The frozen model's weights |

In [ ]:
import os
import sys
import tempfile
from pathlib import Path

os.environ.setdefault('MPLCONFIGDIR', str(Path(tempfile.gettempdir()) / 'sgm_matplotlib_cache'))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

EXPERIMENT_DIR = next(p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents) if (p / 'experiment_common').is_dir()) / 'selection_rule_comparison'
if EXPERIMENT_DIR.name != 'selection_rule_comparison':
    EXPERIMENT_DIR = EXPERIMENT_DIR / 'selection_rule_comparison'
if str(EXPERIMENT_DIR.parent) not in sys.path:
    sys.path.insert(0, str(EXPERIMENT_DIR.parent))

from selection_rule_comparison.archive import beyond_top_degree as BT

RUN_EXPERIMENTS = True                                      # False: only analyse saved results
ARMS_TO_RUN = ['percolation', 'goat', 'goat_other', 'faq']  # run in this order; remove any to skip it
JOBS = 3                                                    # pairs run in parallel

PATH_COLORS = {'pair_mass': '#1F77B4', 'uniqueness': '#2CA02C', 'model': '#E377C2'}
COLORS = {'top_degree': '#222222', 'random': '#9E9E9E', 'witness_all': '#8C564B',
          'hub_free_top_degree': '#FF7F0E', 'hub_free_distinguishing': '#9467BD', 'clustered': '#9ECAE1',
          'pair_mass_2j': '#9ECAE1', 'pair_mass_4j': '#6BAED6', 'pair_mass_8j': '#3182BD',
          'pair_mass_16j': '#2171B5', 'pair_mass_all': '#08519C',
          'uniqueness_2j': '#A1D99B', 'uniqueness_8j': '#41AB5D', 'uniqueness_all': '#006D2C',
          'model_2j': '#F4A6D7', 'model_8j': '#C51B8A',
          'lookalike_pair_mass_all': '#08519C', 'lookalike_uniqueness_all': '#006D2C'}
ARMS_PLOTTED = ['percolation', 'goat', 'goat_other', 'faq']


def models_with_data(frame, arm):
    present = set(frame.loc[frame['arm'] == arm, 'model'])
    return [m for m in BT.ARM_MODELS[arm] if m in present]


def interval(stats):
    return [[stats['mean'] - stats['low']], [stats['high'] - stats['mean']]]

## Run

Progress prints below, one line per pair, with an estimate before it starts. Rerunning skips pairs that are already done, so a stopped run resumes. To run in a terminal instead, from the repository root: `python -m selection_rule_comparison.beyond_top_degree` (add `--arms percolation goat` to run only some arms).

In [ ]:
if RUN_EXPERIMENTS:
    BT.run_all(arms=ARMS_TO_RUN, jobs=JOBS)
display(BT.status())

## 1. Checks

- **Top degree, rerun** on pairs 0, 1, 100 and 101 with round 1's settings. Every run should match the stored one: same success and same accuracy.
- **Frozen model at 2j** should pick exactly the seeds of the 21 Oct pilot's weighted selector, with the same weights.
- Each pair's hashes are checked when it is regenerated; a mismatch stops the run.

In [ ]:
print('Top degree rerun against the stored runs')
display(BT.check_against_stored())
print('Frozen model at 2j against the pilot')
display(BT.check_model_against_pilot())

evaluations = BT.load_evaluations()
queries = BT.queries_needed(evaluations)
curves = BT.success_curves(evaluations)
arms_present = [a for a in ARMS_PLOTTED if (evaluations['arm'] == a).any()]
pairs_done = (evaluations[~evaluations['rule'].isin(['top_degree', 'random'])]
              .groupby(['arm', 'model'])['pair_index'].nunique().rename('pairs with new rules'))
display(pairs_done.to_frame())

## 2. Where each rule's seeds sit

Each point is a rule at the middle budget (means over pairs).
- **x:** share of its seeds that top degree also picks. Top degree is at 100% on the left; moving right means breaking away from top degree.
- **y:** the feature, pair mass on top and uniqueness below.

Lines follow each objective's path: top degree → 2j → … → all nodes. Hollow circles are the look-alikes, grey is random and the brown diamond is the witness score. A path that moves right while going up gets more of its feature by leaving top degree.

In [ ]:
def seed_map(arm):
    algorithm = BT.ARM_ALGORITHM[arm]
    models = models_with_data(evaluations, arm)
    features = [('f_pair_mass', 'pair mass'), (BT.UNIQUENESS_FEATURE[algorithm], 'uniqueness')]
    fig, axes = plt.subplots(2, len(models), figsize=(4.6 * len(models), 7.4), squeeze=False)
    shown = set()
    for column, model in enumerate(models):
        profile = BT.seed_profile(evaluations, arm, model)
        x = 100 * profile['top_degree_overlap']
        for row, (feature, name) in enumerate(features):
            ax = axes[row, column]
            for objective, rules in BT.PATHS.items():
                path = [r for r in ['top_degree', *rules] if r in profile.index]
                if len([r for r in path if r not in ('top_degree', 'pair_mass_2j')]) == 0:
                    continue
                ax.plot(x[path], profile.loc[path, feature], marker='o', markersize=5, color=PATH_COLORS[objective])
                shown.add(objective)
            for rule in ['lookalike_pair_mass_all', 'lookalike_uniqueness_all']:
                if rule in profile.index:
                    ax.scatter(x[rule], profile.loc[rule, feature], s=70, facecolors='white',
                               edgecolors=COLORS[rule], linewidths=1.6, zorder=3)
                    shown.add('lookalike')
            for rule, marker in [('top_degree', 'o'), ('random', 'o'), ('witness_all', 'D')]:
                if rule in profile.index:
                    ax.scatter(x[rule], profile.loc[rule, feature], s=60, color=COLORS[rule], marker=marker, zorder=4)
                    shown.add(rule)
            ax.set_xlim(105, -5)
            if row == 0:
                ax.set_title(f"{BT.MODEL_LABELS[model]} ({int(profile['budget'].iloc[0])} seeds)")
            else:
                ax.set_xlabel('seeds shared with top degree (%)')
            if column == 0:
                ax.set_ylabel(name)
    handles = [Line2D([0], [0], color=PATH_COLORS[o], marker='o', label=BT.OBJECTIVE_LABELS[o])
               for o in BT.PATHS if o in shown]
    handles += [Line2D([0], [0], color=COLORS[r], marker=m, linestyle='', label=label)
                for r, m, label in [('top_degree', 'o', 'top degree'), ('random', 'o', 'random'),
                                    ('witness_all', 'D', 'witness score')] if r in shown]
    if 'lookalike' in shown:
        handles.append(Line2D([0], [0], color='#555555', marker='o', markerfacecolor='white', linestyle='',
                              label='look-alike'))
    fig.legend(handles=handles, loc='lower center', ncol=len(handles), frameon=False)
    fig.suptitle(BT.ARM_LABELS[arm])
    plt.tight_layout(rect=(0, 0.05, 1, 0.97))
    plt.show()


for arm in [a for a in arms_present if a != 'goat_other']:  # goat_other uses GOAT's seeds
    seed_map(arm)

## 3. The paths (main result)

Mean queries needed until the first success, as each objective moves away from top degree (lower is better).
- Bars are 95% bootstrap intervals over pairs.
- The dashed grey line is random.
- Hollow circles at "all nodes" are the look-alikes, and the brown diamond is the witness score.
- A pair that never succeeds counts as one step past the largest budget.

Percolation on ER and PAPER pools the 20 round-1 pairs with the 50 fresh ones. The readings in the introduction apply.

In [ ]:
OFFSETS = {'pair_mass': -0.1, 'uniqueness': 0.0, 'model': 0.1}


def path_plot(arm):
    models = models_with_data(queries, arm)
    summaries = {m: BT.path_summary(queries, arm, m) for m in models}
    used = {p for s in summaries.values() if not s.empty for p in s['pool']} | {'1j', 'all'}
    pools = [p for p in BT.POOL_ORDER if p in used]
    position = {p: i for i, p in enumerate(pools)}
    fig, axes = plt.subplots(1, len(models), figsize=(4.8 * len(models), 4.4), squeeze=False)
    shown = set()
    for ax, model in zip(axes[0], models):
        summary = summaries[model]
        for objective, part in summary.groupby('objective', sort=False):
            xs = [position[p] + (0 if p == '1j' else OFFSETS[objective]) for p in part['pool']]
            ax.errorbar(xs, part['mean'], yerr=[part['mean'] - part['low'], part['high'] - part['mean']],
                        marker='o', markersize=5, color=PATH_COLORS[objective], capsize=3, linewidth=1.6)
            shown.add(objective)
        for rule, objective in [('lookalike_pair_mass_all', 'pair_mass'), ('lookalike_uniqueness_all', 'uniqueness')]:
            stats = BT.mean_queries(queries, arm, model, rule)
            if stats['pairs']:
                ax.errorbar([position['all'] + OFFSETS[objective] + 0.22], [stats['mean']], yerr=interval(stats),
                            marker='o', markersize=7, markerfacecolor='white', color=PATH_COLORS[objective],
                            capsize=3, linestyle='none')
                shown.add('lookalike')
        witness = BT.mean_queries(queries, arm, model, 'witness_all')
        if witness['pairs']:
            ax.errorbar([position['all'] - 0.25], [witness['mean']], yerr=interval(witness), marker='D', markersize=6,
                        color=COLORS['witness_all'], capsize=3, linestyle='none')
            shown.add('witness')
        random = BT.mean_queries(queries, arm, model, 'random')
        if random['pairs']:
            ax.axhline(random['mean'], color=COLORS['random'], linestyle='--', linewidth=1.3)
            shown.add('random')
        top = BT.mean_queries(queries, arm, model, 'top_degree')
        ax.set_xticks(range(len(pools)))
        ax.set_xticklabels([BT.POOL_LABELS[p] for p in pools])
        ax.set_title(f"{BT.MODEL_LABELS[model]} ({top['pairs']} pairs)")
        ax.set_xlabel('candidate pool for seed j')
    axes[0, 0].set_ylabel('queries needed (lower is better)')
    handles = [Line2D([0], [0], color=PATH_COLORS[o], marker='o', label=BT.OBJECTIVE_LABELS[o])
               for o in BT.PATHS if o in shown]
    if 'lookalike' in shown:
        handles.append(Line2D([0], [0], color='#555555', marker='o', markerfacecolor='white', linestyle='',
                              label='look-alike'))
    if 'witness' in shown:
        handles.append(Line2D([0], [0], color=COLORS['witness_all'], marker='D', linestyle='', label='witness score'))
    if 'random' in shown:
        handles.append(Line2D([0], [0], color=COLORS['random'], linestyle='--', label='random'))
    fig.legend(handles=handles, loc='lower center', ncol=len(handles), frameon=False)
    fig.suptitle(BT.ARM_LABELS[arm])
    plt.tight_layout(rect=(0, 0.07, 1, 0.95))
    plt.show()


for arm in [a for a in arms_present if a != 'goat_other']:
    path_plot(arm)

## 4. Degree level or feature?

For each all-node rule, the queries it saves against random (positive = fewer queries), split into two parts:
- **degree level** (grey): random − look-alike, what choosing nodes of those degrees buys;
- **feature** (colour): look-alike − rule, what the feature adds at that degree level.

The two parts add up to the rule's total saving. Bars are means over pairs with 95% bootstrap intervals; the table gives the numbers. GOAT is shown at both regularisers.

In [ ]:
split_rows = []
for arm in arms_present:
    for model in models_with_data(queries, arm):
        for rule in BT.LOOKALIKE_SOURCES:
            split = BT.gain_split(queries, arm, model, rule)
            if split:
                split_rows.append({'arm': arm, 'model': model, 'rule': rule, 'pairs': split['pairs'],
                                   **{name: value for name, value in split.items() if name != 'pairs'}})

if split_rows:
    split_frame = pd.DataFrame(split_rows)
    arms_split = [a for a in arms_present if (split_frame['arm'] == a).any()]
    fig, axes = plt.subplots(1, len(arms_split), figsize=(4.2 * len(arms_split) + 1, 4.4), squeeze=False)
    for ax, arm in zip(axes[0], arms_split):
        part = split_frame[split_frame['arm'] == arm].reset_index(drop=True)
        for i, row in part.iterrows():
            objective = 'pair_mass' if row['rule'].startswith('pair_mass') else 'uniqueness'
            for offset, name, color in [(-0.19, 'degree level (random - look-alike)', '#BDBDBD'),
                                        (0.19, 'feature (look-alike - rule)', PATH_COLORS[objective])]:
                mean, low, high = row[name]
                ax.bar(i + offset, mean, width=0.36, color=color)
                ax.errorbar(i + offset, mean, yerr=[[mean - low], [high - mean]], color='black', capsize=3,
                            linewidth=1)
        ax.axhline(0, color='black', linewidth=0.8)
        ax.set_xticks(range(len(part)))
        ax.set_xticklabels([f"{BT.MODEL_LABELS[m]}\n{BT.OBJECTIVE_LABELS[r.rsplit('_', 1)[0]]}"
                            for m, r in zip(part['model'], part['rule'])], fontsize=8)
        ax.set_title(BT.ARM_LABELS[arm])
    axes[0, 0].set_ylabel('queries saved against random')
    fig.legend(handles=[Patch(color='#BDBDBD', label='degree level'),
                        Patch(color=PATH_COLORS['pair_mass'], label='feature: pair mass'),
                        Patch(color=PATH_COLORS['uniqueness'], label='feature: uniqueness')],
               loc='lower center', ncol=3, frameon=False)
    plt.tight_layout(rect=(0, 0.08, 1, 1))
    plt.show()

    def fmt(value):
        mean, low, high = value
        return f'{mean:+.2f} [{low:+.2f}, {high:+.2f}]'

    table = split_frame.assign(arm=split_frame['arm'].map(BT.ARM_LABELS),
                               model=split_frame['model'].map(BT.MODEL_LABELS),
                               rule=split_frame['rule'].map(BT.rule_label))
    for column in ['degree level (random - look-alike)', 'feature (look-alike - rule)', 'total (random - rule)']:
        table[column] = table[column].map(fmt)
    display(table.set_index(['arm', 'model', 'rule']))
else:
    print('No all-node rule with a look-alike has finished yet.')

## 5. Success curves

Share of pairs that succeed at each budget, for the 2j and all-node rules (the 8j rules are in the path plots). Look-alike draws are averaged, and look-alikes are dashed in their rule's colour. Darker shades are wider pools.

In [ ]:
CURVE_RULES = ['top_degree', 'random', 'hub_free_top_degree', 'pair_mass_2j', 'pair_mass_all',
               'lookalike_pair_mass_all', 'uniqueness_2j', 'uniqueness_all', 'lookalike_uniqueness_all',
               'model_2j', 'witness_all']  # the 8j rules are in the path plots

for arm in arms_present:
    models = models_with_data(curves, arm)
    fig, axes = plt.subplots(1, len(models), figsize=(4.8 * len(models), 4.0), sharey=True, squeeze=False)
    labels = {}
    for ax, model in zip(axes[0], models):
        data = curves[(curves['arm'] == arm) & (curves['model'] == model)]
        for rule in CURVE_RULES:
            part = data[data['rule'] == rule].sort_values('budget')
            if part.empty:
                continue
            line, = ax.plot(part['budget'], part['success'], marker='o', markersize=3.5, color=COLORS[rule],
                            linestyle='--' if rule.startswith('lookalike_') else '-', linewidth=1.5)
            labels[rule] = line
        ax.set_title(BT.MODEL_LABELS[model])
        ax.set_xlabel('seeds (queries)')
        ax.set_ylim(-0.03, 1.03)
    axes[0, 0].set_ylabel('success rate')
    order = [r for r in CURVE_RULES if r in labels]
    fig.legend([labels[r] for r in order], [BT.rule_label(r) for r in order], loc='lower center',
               ncol=min(5, len(order)), frameon=False, fontsize=8)
    fig.suptitle(BT.ARM_LABELS[arm])
    rows = int(np.ceil(len(order) / 5))
    plt.tight_layout(rect=(0, 0.06 * rows + 0.02, 1, 0.95))
    plt.show()

## 6. Pair-by-pair comparisons

Each new rule is compared with top degree and random. All-node rules are also compared with their look-alike and with round 1's best rule: clustered for percolation, hub-free distinguishing for GOAT on PAPER and Chung–Lu, distinguishing otherwise.
- **fewer / same / more:** pairs where the rule needs fewer, the same or more queries than the other one.
- **p:** sign test of fewer against more.
- **mean change:** the rule minus the other, with a 95% bootstrap interval. Negative means the rule needs fewer queries.

In [ ]:
for arm in arms_present:
    table = BT.comparison_table(queries, arm)
    if not table.empty:
        print(BT.ARM_LABELS[arm])
        display(table.round(3))

## 7. GOAT at both regularisers

Round 1 ran GOAT at 100 on ER and at 300 on PAPER and Chung–Lu. The temperature audit found that the hub result flips between 100 and 300, so the same seed sets are run here at the other value.
- The bars compare mean queries needed at the two values.
- The table repeats the key comparisons at each value. A look-alike has the same degrees as its rule, so it runs at roughly the same temperature. If the uniqueness rule beats its look-alike at both values, its gain is not a temperature effect.

In [ ]:
SHORT_LABELS = {'top_degree': 'top\ndegree', 'random': 'random', 'hub_free_top_degree': 'hub-free\ntop degree',
                'pair_mass_all': 'pair mass\nall nodes', 'uniqueness_all': 'uniqueness\nall nodes',
                'lookalike_pair_mass_all_1': 'look-alike\npair mass', 'lookalike_uniqueness_all_1': 'look-alike\nuniqueness'}
models = models_with_data(queries, 'goat_other')
if models:
    fig, axes = plt.subplots(1, len(models), figsize=(5.2 * len(models), 4.4), squeeze=False)
    for ax, model in zip(axes[0], models):
        table = BT.regularizer_table(queries, model)
        if table.empty:
            ax.set_visible(False)
            continue
        columns = sorted([c for c in table.columns if c.startswith('regulariser')], key=lambda c: int(c.split()[-1]))
        x = np.arange(len(table))
        for j, column in enumerate(columns):
            ax.bar(x + (j - 0.5) * 0.38, table[column], width=0.38, color=['#BDBDBD', '#525252'][j], label=column)
        ax.set_xticks(x)
        ax.set_xticklabels([SHORT_LABELS[rule] for rule in table.index], fontsize=8)
        ax.set_title(f"{BT.MODEL_LABELS[model]} ({int(table['pairs'].min())} pairs)")
        ax.legend(frameon=False, fontsize=8)
    axes[0, 0].set_ylabel('queries needed (lower is better)')
    plt.tight_layout()
    plt.show()

    CONTRASTS = [('uniqueness_all', 'lookalike_uniqueness_all_1'), ('pair_mass_all', 'lookalike_pair_mass_all_1'),
                 ('uniqueness_all', 'top_degree'), ('hub_free_top_degree', 'top_degree'), ('random', 'top_degree')]
    rows = []
    for model in models:
        for arm in ['goat', 'goat_other']:
            for first, second in CONTRASTS:
                result = BT.compare(queries, arm, model, first, second)
                if result:
                    rows.append({'model': BT.MODEL_LABELS[model], 'regulariser': BT.regularizer(arm, model),
                                 'comparison': f'{BT.rule_label(first)} vs {BT.rule_label(second)}', **result})
    display(pd.DataFrame(rows).set_index(['model', 'comparison', 'regulariser']).sort_index().round(3))
else:
    print('The other-regulariser arm has not run yet.')

## 8. The frozen model's weights

These are the weights the frozen-model rule maximises: log-odds per standard deviation of the training random sets. Each comes from a fixed-pair collection, so one pair at one budget. `same as pilot` checks them against the weights saved by the 21 Oct pilot.

In [ ]:
display(BT.weights_table().round(3))